# 16. Normalized BM25 + Vector RRF weight ablation

이 노트북은 **Codex coder agent가 `skn25` 환경에서 실행한 개발셋 전용 실험**이다. 기존 사용자 실행 노트북 provenance와 구분한다. 기존 30개 개발 질의만 사용하며 독립 holdout이나 운영 일반화를 주장하지 않는다.

## 사전 고정 계약

- Vector:Normalized BM25 가중치: `0.4:0.6`, `0.5:0.5`, `0.6:0.4`, `0.7:0.3`, `0.8:0.2`
- BM25: `k1=1.5`, `b=0.75`; RRF: `k=60`, component depth `50`
- 점수: `vector_weight/(60+vector_rank) + bm25_weight/(60+bm25_rank)`; rank 50 밖 component 기여는 0
- 정렬: total score 원값 내림차순, 동점은 chunk ID lexical 오름차순. 정렬 전 반올림 없음.
- Primary: evidence 20개 MRR@5. Baseline은 `0.5:0.5`.
- 후보 자격: MRR delta `>= +0.025`이면서 evidence Strict Hit@3/Recall@5/nDCG@5/Card Hit@3와 card-group Card Hit@3가 모두 비회귀.
- 후보가 없으면 baseline 유지.
- 복수 후보 tie-break: tolerance `1e-12`로 evidence MRR, nDCG, Recall, Strict, Card를 높은 순서로 비교하고, 그다음 `|vector_weight-0.5|`가 작은 순서, vector weight가 큰 순서로 비교한다.

실행은 `conda run -n skn25` 아래 `nbclient 0.10.4`의 `NotebookClient(timeout=900, kernel_name='python3')`로 in-place 수행한다. 실행 드라이버는 notebook 저장 후 manifest의 serialized notebook raw SHA-256만 최종화한다.

In [1]:
from __future__ import annotations

import csv
import gc
import hashlib
import json
import math
import os
import re
import shutil
import sqlite3
import tempfile
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from decimal import Decimal
from functools import cmp_to_key
from pathlib import Path

import chromadb
import numpy as np
from chromadb.config import Settings

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl').is_file())
SOURCE_ROOT = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
MORPH_ROOT = PROJECT_ROOT / 'notebooks/data/15_korean_morphology_retrieval_ablation'
OUTPUT_ROOT = PROJECT_ROOT / 'notebooks/data/16_normalized_rrf_weight_ablation'
NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks/16_normalized_rrf_weight_ablation.ipynb'
CHROMA_ROOT = SOURCE_ROOT / 'chroma'
EMBEDDING_MODEL = 'text-embedding-3-small'
RAW_TOKEN = re.compile(r'[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*', re.IGNORECASE)
BM25_K1, BM25_B = 1.5, 0.75
RRF_K, RRF_DEPTH = 60, 50
WEIGHT_PAIRS = ((0.4, 0.6), (0.5, 0.5), (0.6, 0.4), (0.7, 0.3), (0.8, 0.2))
BASELINE_VECTOR_WEIGHT = 0.5
PRIMARY_DELTA = 0.025
TOLERANCE = 1e-12
CANDIDATE_DEPTHS = (10, 20, 50)


def configuration_name(vector_weight, bm25_weight):
    return f'vector_{vector_weight:.1f}_bm25_{bm25_weight:.1f}'


CONFIGURATIONS = {configuration_name(vector_weight, bm25_weight): (vector_weight, bm25_weight) for vector_weight, bm25_weight in WEIGHT_PAIRS}
BASELINE_CONFIGURATION = configuration_name(0.5, 0.5)
SELECTION_CONTRACT = {
    'primary': {'group': 'evidence', 'metric': 'mrr_at_5', 'minimum_delta': PRIMARY_DELTA},
    'guardrails': ['evidence.strict_evidence_hit_at_3', 'evidence.recall_at_5', 'evidence.ndcg_at_5', 'evidence.card_hit_at_3', 'card.card_hit_at_3'],
    'tie_break': ['mrr_at_5_desc', 'ndcg_at_5_desc', 'recall_at_5_desc', 'strict_evidence_hit_at_3_desc', 'card_hit_at_3_desc', 'distance_to_0.5_asc', 'vector_weight_desc'],
    'tolerance': TOLERANCE,
    'fallback': BASELINE_CONFIGURATION,
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({'agent_provenance': 'Codex coder agent', 'environment': 'skn25', 'configurations': CONFIGURATIONS, 'selection_contract': SELECTION_CONTRACT})

{'agent_provenance': 'Codex coder agent', 'environment': 'skn25', 'configurations': {'vector_0.4_bm25_0.6': (0.4, 0.6), 'vector_0.5_bm25_0.5': (0.5, 0.5), 'vector_0.6_bm25_0.4': (0.6, 0.4), 'vector_0.7_bm25_0.3': (0.7, 0.3), 'vector_0.8_bm25_0.2': (0.8, 0.2)}, 'selection_contract': {'primary': {'group': 'evidence', 'metric': 'mrr_at_5', 'minimum_delta': 0.025}, 'guardrails': ['evidence.strict_evidence_hit_at_3', 'evidence.recall_at_5', 'evidence.ndcg_at_5', 'evidence.card_hit_at_3', 'card.card_hit_at_3'], 'tie_break': ['mrr_at_5_desc', 'ndcg_at_5_desc', 'recall_at_5_desc', 'strict_evidence_hit_at_3_desc', 'card_hit_at_3_desc', 'distance_to_0.5_asc', 'vector_weight_desc'], 'tolerance': 1e-12, 'fallback': 'vector_0.5_bm25_0.5'}}


## Allowed inputs and immutable-state capture

입력은 명시된 13번·15번 notebook/artifact, embedding usage에 기록된 정확한 6개 cache, `chroma/**`로 제한한다. 원본 SQLite는 URI read-only로만 열고 원본 Chroma에는 `PersistentClient`를 연결하지 않는다. 모든 허용 입력과 Chroma tree/count를 실행 전후 비교한다.

Chroma tree hash는 deterministic directory preorder(루트 파일 우선, 각 디렉터리의 파일명·하위 디렉터리 정렬)로 만든 상대경로→raw SHA-256 map을 단일 근거로 계산하고, 15번 frozen 실행 후 hash와 즉시 대조한다.

In [2]:
def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def atomic_json(path, value):
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        json.dump(value, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_csv(path, rows):
    rows = list(rows)
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temporary, path)


def files_under(root):
    # Deterministic directory preorder: root files first, then sorted subdirectories.
    paths = []
    for directory, subdirectories, names in os.walk(root):
        subdirectories.sort()
        paths.extend(Path(directory) / name for name in sorted(names))
    return paths


def file_hash_map(root):
    return {path.relative_to(root).as_posix(): sha256_file(path) for path in files_under(root)}


def tree_hash_from_file_hash_map(hashes):
    digest = hashlib.sha256()
    for relative_path, raw_sha256 in hashes.items():
        digest.update(relative_path.encode())
        digest.update(bytes.fromhex(raw_sha256))
    return digest.hexdigest()


def tree_hash(root):
    hashes = file_hash_map(root)
    digest = tree_hash_from_file_hash_map(hashes)
    assert digest == tree_hash_from_file_hash_map(dict(hashes))
    return digest


def assert_current_chroma(expected_hash_map, expected_tree_hash, stage):
    current_hash_map = file_hash_map(CHROMA_ROOT)
    current_tree_hash = tree_hash_from_file_hash_map(current_hash_map)
    assert current_hash_map == expected_hash_map, stage
    assert current_tree_hash == expected_tree_hash, stage
    assert current_tree_hash == tree_hash(CHROMA_ROOT), stage
    return {'stage': stage, 'files': len(current_hash_map), 'tree_hash': current_tree_hash}


SOURCE_FILES = {
    'contract_notebook_13': PROJECT_ROOT / 'notebooks/13_hierarchical_chunking_retrieval.ipynb',
    'chunks': SOURCE_ROOT / 'chunks.jsonl',
    'retrieval_per_query': SOURCE_ROOT / 'retrieval_per_query.csv',
    'normalization_per_query': SOURCE_ROOT / 'retrieval_search_normalization_per_query.csv',
    'normalization_summary': SOURCE_ROOT / 'retrieval_search_normalization_summary.json',
    'embedding_usage': SOURCE_ROOT / 'embedding_usage.json',
    'input_manifest': SOURCE_ROOT / 'input_manifest.json',
    'index_manifest': SOURCE_ROOT / 'index_manifest.json',
    'morph_notebook_15': PROJECT_ROOT / 'notebooks/15_korean_morphology_retrieval_ablation.ipynb',
    'morph_per_query_15': MORPH_ROOT / 'morphology_ablation_per_query.csv',
    'morph_summary_15': MORPH_ROOT / 'morphology_ablation_summary.json',
    'morph_manifest_15': MORPH_ROOT / 'morphology_run_manifest.json',
}
morphology_summary_15 = json.loads(SOURCE_FILES['morph_summary_15'].read_text(encoding='utf-8'))
FROZEN_EXPECTED_CHROMA_TREE_HASH = morphology_summary_15['integrity']['chroma_tree_hash_after']
assert FROZEN_EXPECTED_CHROMA_TREE_HASH == '255dd5d9cdd84065a0d75a73b6b3b9bf09962d02797c4103352a909d4efc15c5'

embedding_usage = json.loads(SOURCE_FILES['embedding_usage'].read_text(encoding='utf-8'))
cache_fingerprints = [batch['batch_fingerprint'] for batch in embedding_usage['batches']]
assert cache_fingerprints == [
    '3c81bda6bc8b1e69ca305b0dcc219203baeede6797e91045dbde36244babc3b4',
    '335a583c624bd1fe61aec74282af4e1462af20962c96147ada05ed5a29605fba',
    '93cd3f7a2ca8c9e070421705a575465a92f29966aebae38a583c2da60a6ee97f',
    'b6ae13c54bbba3d250e83c39da8673647b077dfc91f671565b4cef016593fbf2',
    '8433343533d6e8ae60f40c42d7e4a0b43b751bb36e26532e738ff51eac4a9ae0',
    'cb5d9e87e091be498d91fa51d46619ca60c7fe5314b27a6686b017cb8c304e55',
]
CACHE_FILES = {
    f'embedding_cache_{index + 1}': SOURCE_ROOT / 'embedding_cache' / EMBEDDING_MODEL / f'{fingerprint}.npz'
    for index, fingerprint in enumerate(cache_fingerprints)
}
CHROMA_FILES = {
    f'chroma:{path.relative_to(CHROMA_ROOT).as_posix()}': path for path in files_under(CHROMA_ROOT)
}
ALL_INPUT_FILES = {**SOURCE_FILES, **CACHE_FILES, **CHROMA_FILES}
input_hashes_before = {name: sha256_file(path) for name, path in ALL_INPUT_FILES.items()}
chroma_file_hashes_before = file_hash_map(CHROMA_ROOT)
chroma_hash_map_from_input_hashes = {
    name.removeprefix('chroma:'): input_hashes_before[name] for name in CHROMA_FILES
}
assert chroma_hash_map_from_input_hashes == chroma_file_hashes_before
chroma_tree_hash_before = tree_hash_from_file_hash_map(chroma_file_hashes_before)
assert chroma_tree_hash_before == tree_hash(CHROMA_ROOT)
assert chroma_tree_hash_before == FROZEN_EXPECTED_CHROMA_TREE_HASH
integrity_stage_checks = [assert_current_chroma(chroma_file_hashes_before, FROZEN_EXPECTED_CHROMA_TREE_HASH, 'after_input_hashes_before')]

PERFORMANCE_RESULT_FILES = {
    'rrf_weight_ablation_per_query.csv': OUTPUT_ROOT / 'rrf_weight_ablation_per_query.csv',
    'rrf_weight_ablation_summary.csv': OUTPUT_ROOT / 'rrf_weight_ablation_summary.csv',
    'rrf_weight_candidate_recall.csv': OUTPUT_ROOT / 'rrf_weight_candidate_recall.csv',
    'rrf_weight_union_coverage.csv': OUTPUT_ROOT / 'rrf_weight_union_coverage.csv',
    'rrf_weight_candidates.csv': OUTPUT_ROOT / 'rrf_weight_candidates.csv',
}
FROZEN_SEVEN_WEIGHT_PERFORMANCE_HASHES = {
    'rrf_weight_ablation_per_query.csv': '7eff7aed73fd3c2957192dcd62207ddae8c0c0d1ca35f35267287094f7fd3ea2',
    'rrf_weight_ablation_summary.csv': 'bdf10d302f9071e054c08e429d0f88795fb5e48b38b2a42b817951f55b1effb0',
    'rrf_weight_candidate_recall.csv': '13b5423b9370ae824f3264329cd436e412a9a1acc0db0db1533bd18f3f7e49d8',
    'rrf_weight_union_coverage.csv': '826a67350a5996cf8b8609d8a55777f439db2674bff955c06a00244c9cac896a',
    'rrf_weight_candidates.csv': '051d46b72880f583a03ded1d21af0cb88486230c99366d99b4be08606f8e91d8',
}
pre_reexecution_performance_hashes = {name: sha256_file(result_path) for name, result_path in PERFORMANCE_RESULT_FILES.items()}
pre_reexecution_hash_exact_by_file = {
    name: pre_reexecution_performance_hashes[name] == frozen_hash
    for name, frozen_hash in FROZEN_SEVEN_WEIGHT_PERFORMANCE_HASHES.items()
}
assert pre_reexecution_hash_exact_by_file['rrf_weight_ablation_per_query.csv']
assert pre_reexecution_hash_exact_by_file['rrf_weight_ablation_summary.csv']

database = (CHROMA_ROOT / 'chroma.sqlite3').resolve()
with sqlite3.connect(f'file:{database}?mode=ro', uri=True) as connection:
    collection_rows = connection.execute('SELECT id, name, config_json_str FROM collections').fetchall()
    collection_metadata = connection.execute('SELECT key, str_value, int_value, float_value, bool_value FROM collection_metadata').fetchall()
    segment_metadata = connection.execute('SELECT key, str_value, int_value, float_value, bool_value FROM segment_metadata').fetchall()
    chroma_count_before = connection.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]
assert len(collection_rows) == 1 and chroma_count_before == 327
collection_id, collection_name, collection_config_raw = collection_rows[0]
collection_config = json.loads(collection_config_raw)
distance_override_rows = [row for row in [*collection_metadata, *segment_metadata] if row[0] in {'hnsw:space', 'space', 'distance'}]
assert collection_config == {} and not distance_override_rows

chunks = [json.loads(line) for line in SOURCE_FILES['chunks'].read_text(encoding='utf-8').splitlines()]
chunk_by_id = {chunk['id']: chunk for chunk in chunks}
baseline_rows = list(csv.DictReader(SOURCE_FILES['retrieval_per_query'].open(encoding='utf-8')))
saved_rows = list(csv.DictReader(SOURCE_FILES['normalization_per_query'].open(encoding='utf-8')))
saved_summary = json.loads(SOURCE_FILES['normalization_summary'].read_text(encoding='utf-8'))
evaluation_by_id = {}
for row in baseline_rows:
    if row['method'] == 'keyword':
        evaluation_by_id[row['query_id']] = {
            'query_id': row['query_id'], 'query': row['query'], 'category': row['category'],
            'expected_card': row['expected_card'], 'expected_level': row['expected_level'],
            'required_terms': json.loads(row['required_terms']),
        }
assert len(chunks) == 327 and len(evaluation_by_id) == 30
assert sum(item['expected_level'] == 'card' for item in evaluation_by_id.values()) == 10
assert sum(item['expected_level'] != 'card' for item in evaluation_by_id.values()) == 20
search_queries = {query_id: item['query'] for query_id, item in evaluation_by_id.items()}
print({'chunks': len(chunks), 'queries': len(search_queries), 'evidence': 20, 'card': 10, 'input_files': len(ALL_INPUT_FILES), 'chroma_files': len(chroma_file_hashes_before), 'chroma_tree_hash': chroma_tree_hash_before, 'frozen_expected_hash_from_15': FROZEN_EXPECTED_CHROMA_TREE_HASH, 'tree_hash_from_file_map_exact': True, 'chroma_count': chroma_count_before, 'sqlite_config': collection_config, 'pre_reexecution_top5_metric_hashes_exact': True, 'pre_reexecution_candidate_hashes_exact': all(pre_reexecution_hash_exact_by_file[name] for name in ('rrf_weight_candidate_recall.csv', 'rrf_weight_union_coverage.csv', 'rrf_weight_candidates.csv'))})

{'chunks': 327, 'queries': 30, 'evidence': 20, 'card': 10, 'input_files': 23, 'chroma_files': 5, 'chroma_tree_hash': '255dd5d9cdd84065a0d75a73b6b3b9bf09962d02797c4103352a909d4efc15c5', 'frozen_expected_hash_from_15': '255dd5d9cdd84065a0d75a73b6b3b9bf09962d02797c4103352a909d4efc15c5', 'tree_hash_from_file_map_exact': True, 'chroma_count': 327, 'sqlite_config': {}, 'pre_reexecution_top5_metric_hashes_exact': True, 'pre_reexecution_candidate_hashes_exact': True}


## Normalized BM25 and verified cached vector rank

기존 `search_tokens`와 BM25를 그대로 복제한다. Cache 357개를 hash/fingerprint/dimension/dtype/finite까지 검증한다. SQLite config 진단 후 NumPy squared-L2 full rank를 먼저 시도하고, published vector top-5가 30/30이 아니면 Chroma tree를 `/tmp`에 byte-copy한 snapshot에만 연결해 depth 50 rank를 얻는다.

In [3]:
normalized_text = lambda value: ' '.join(unicodedata.normalize('NFKC', str(value)).lower().split())


def canonical_decimal(value):
    rendered = format(Decimal(str(value).replace(',', '')).normalize(), 'f')
    rendered = rendered.rstrip('0').rstrip('.') if '.' in rendered else rendered
    return '0' if rendered in {'', '-0'} else rendered


def search_tokens(value):
    text = normalized_text(value)
    tokens = list(RAW_TOKEN.findall(text))
    for run in re.findall(r'[가-힣](?:[가-힣 ]{0,38}[가-힣])?', text):
        joined = run.replace(' ', '')
        for size in (2, 3, 4):
            tokens.extend(f'ko{size}_{joined[index:index + size]}' for index in range(max(0, len(joined) - size + 1)))
    consumed = []
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*만\s*(\d[\d,]*(?:\.\d+)?)\s*천\s*원', text):
        amount = Decimal(match.group(1).replace(',', '')) * 10000 + Decimal(match.group(2).replace(',', '')) * 1000
        tokens.append(f'money_krw_{canonical_decimal(amount)}')
        consumed.append(match.span())
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*(만|천)?\s*원', text):
        if any(left <= match.start() and match.end() <= right for left, right in consumed):
            continue
        multiplier = {'만': 10000, '천': 1000, None: 1}[match.group(2)]
        amount = Decimal(match.group(1).replace(',', '')) * multiplier
        tokens.append(f'money_krw_{canonical_decimal(amount)}')
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*%', text):
        tokens.append(f'percent_{canonical_decimal(match.group(1))}')
    for match in re.finditer(r'(?:(월|연|년|일)\s*)?(\d[\d,]*(?:\.\d+)?)\s*(회|개월|년|일)', text):
        prefix = match.group(1) or 'none'
        tokens.append(f'period_{prefix}_{canonical_decimal(match.group(2))}_{match.group(3)}')
    return tokens


def bm25_scores(query_tokens, documents):
    tokenized = {identifier: list(tokens) for identifier, tokens in documents.items()}
    document_frequency = Counter(token for tokens in tokenized.values() for token in set(tokens))
    average_length = sum(map(len, tokenized.values())) / len(tokenized) if tokenized else 1.0
    scores = {}
    for identifier, tokens in tokenized.items():
        frequencies, score = Counter(tokens), 0.0
        for token in query_tokens:
            frequency = frequencies[token]
            if frequency:
                inverse_frequency = math.log(1 + (len(tokenized) - document_frequency[token] + 0.5) / (document_frequency[token] + 0.5))
                score += inverse_frequency * frequency * (BM25_K1 + 1) / (frequency + BM25_K1 * (1 - BM25_B + BM25_B * len(tokens) / average_length))
        scores[identifier] = score
    return scores


def rank_scores(scores):
    return [identifier for identifier, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0]))]


normalized_documents = {chunk['id']: search_tokens(chunk['document']) for chunk in chunks}
normalized_queries = {query_id: search_tokens(text) for query_id, text in search_queries.items()}
normalized_bm25 = {query_id: rank_scores(bm25_scores(normalized_queries[query_id], normalized_documents)) for query_id in search_queries}
published_normalized = {row['query_id']: json.loads(row['top5_chunk_ids']) for row in saved_rows if row['configuration'] == 'normalized_bm25'}
assert all(normalized_bm25[query_id][:5] == published_normalized[query_id] for query_id in search_queries)

cached_items = [(f"chunk:{chunk['id']}", chunk['document']) for chunk in chunks] + [(f'query:{query_id}', text) for query_id, text in search_queries.items()]
assert len(cached_items) == embedding_usage['embedding_items'] == 357
vectors_by_key = {}
cache_validation = []
for batch_index, batch_start in enumerate(range(0, len(cached_items), 64)):
    batch = cached_items[batch_start:batch_start + 64]
    hashes = [hashlib.sha256(text.encode()).hexdigest() for _, text in batch]
    fingerprint = hashlib.sha256(canonical_json({'model': EMBEDDING_MODEL, 'hashes': hashes}).encode()).hexdigest()
    assert fingerprint == cache_fingerprints[batch_index]
    with np.load(CACHE_FILES[f'embedding_cache_{batch_index + 1}'], allow_pickle=False) as cached:
        embeddings = cached['embeddings']
        assert cached['hashes'].tolist() == hashes
        assert embeddings.shape == (len(batch), 1536)
        assert embeddings.dtype == np.float32 and np.isfinite(embeddings).all()
        vectors_by_key.update({key: vector.copy() for (key, _), vector in zip(batch, embeddings)})
        cache_validation.append({'batch': batch_index + 1, 'fingerprint': fingerprint, 'items': len(batch), 'dimension': 1536, 'dtype': str(embeddings.dtype), 'finite': True})
assert len(vectors_by_key) == 357

chunk_ids = [chunk['id'] for chunk in chunks]
chunk_vectors = np.stack([vectors_by_key[f'chunk:{identifier}'] for identifier in chunk_ids])
query_vectors = {query_id: vectors_by_key[f'query:{query_id}'] for query_id in search_queries}
published_vector = {row['query_id']: json.loads(row['top5_chunk_ids']) for row in baseline_rows if row['method'] == 'vector'}


def l2_rank(query_vector):
    distances = np.sum((chunk_vectors - query_vector) ** 2, axis=1)
    return [identifier for identifier, _ in sorted(zip(chunk_ids, distances.tolist()), key=lambda item: (item[1], item[0]))]


numpy_vector_rank = {query_id: l2_rank(query_vectors[query_id]) for query_id in search_queries}
numpy_published_top5_exact_queries = sum(numpy_vector_rank[query_id][:5] == published_vector[query_id] for query_id in search_queries)
vector_fallback_used = numpy_published_top5_exact_queries != 30
distance_matches_squared_l2 = True
integrity_stage_checks.append(assert_current_chroma(chroma_file_hashes_before, FROZEN_EXPECTED_CHROMA_TREE_HASH, 'before_snapshot_copy'))
published_baseline_rrf_top5 = {
    row['query_id']: json.loads(row['top5_chunk_ids'])
    for row in saved_rows if row['configuration'] == 'rrf_vector_normalized'
}


def reference_rrf_top5(bm25_ranking, candidate_vector_ranking):
    scores = defaultdict(float)
    for weight, ranking in ((0.5, bm25_ranking[:RRF_DEPTH]), (0.5, candidate_vector_ranking[:RRF_DEPTH])):
        for rank, identifier in enumerate(ranking, 1):
            scores[identifier] += weight / (RRF_K + rank)
    return [identifier for identifier, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0]))][:5]


snapshot_attempt_top5_exact_queries = []
snapshot_attempt_baseline_rrf_exact_queries = []
if vector_fallback_used:
    vector_rank = None
    for snapshot_attempt in range(1, 9):
        integrity_stage_checks.append(assert_current_chroma(chroma_file_hashes_before, FROZEN_EXPECTED_CHROMA_TREE_HASH, f'before_snapshot_copy_attempt_{snapshot_attempt}'))
        chroma_snapshot = tempfile.TemporaryDirectory()
        snapshot_root = Path(chroma_snapshot.name) / 'chroma'
        shutil.copytree(CHROMA_ROOT, snapshot_root)
        integrity_stage_checks.append(assert_current_chroma(chroma_file_hashes_before, FROZEN_EXPECTED_CHROMA_TREE_HASH, f'after_snapshot_copy_attempt_{snapshot_attempt}'))
        snapshot_hash_map_before_client = file_hash_map(snapshot_root)
        assert snapshot_hash_map_before_client == chroma_file_hashes_before
        assert tree_hash_from_file_hash_map(snapshot_hash_map_before_client) == FROZEN_EXPECTED_CHROMA_TREE_HASH
        chroma_client = chromadb.PersistentClient(path=str(snapshot_root), settings=Settings(anonymized_telemetry=False))
        collection = chroma_client.get_collection(collection_name)
        attempted_rank = {}
        attempted_distance_match = True
        for query_id in search_queries:
            response = collection.query(query_embeddings=[query_vectors[query_id].tolist()], n_results=RRF_DEPTH, include=['distances'])
            attempted_rank[query_id] = response['ids'][0]
            l2_by_id = {identifier: float(np.sum((chunk_vectors[index] - query_vectors[query_id]) ** 2)) for index, identifier in enumerate(chunk_ids)}
            attempted_distance_match = attempted_distance_match and np.allclose(response['distances'][0], [l2_by_id[identifier] for identifier in attempted_rank[query_id]], rtol=2e-5, atol=2e-5)
        assert collection.count() == 327 and attempted_distance_match
        attempt_exact = sum(attempted_rank[query_id][:5] == published_vector[query_id] for query_id in search_queries)
        attempt_rrf_exact = sum(
            reference_rrf_top5(normalized_bm25[query_id], attempted_rank[query_id]) == published_baseline_rrf_top5[query_id]
            for query_id in search_queries
        )
        snapshot_attempt_top5_exact_queries.append(attempt_exact)
        snapshot_attempt_baseline_rrf_exact_queries.append(attempt_rrf_exact)
        integrity_stage_checks.append(assert_current_chroma(chroma_file_hashes_before, FROZEN_EXPECTED_CHROMA_TREE_HASH, f'after_snapshot_query_attempt_{snapshot_attempt}'))
        del collection, chroma_client
        gc.collect()
        chroma_snapshot.cleanup()
        if attempt_exact == 30 and attempt_rrf_exact == 30:
            vector_rank = attempted_rank
            distance_matches_squared_l2 = attempted_distance_match
            break
    assert vector_rank is not None, {'vector_top5': snapshot_attempt_top5_exact_queries, 'baseline_rrf_top5': snapshot_attempt_baseline_rrf_exact_queries}
else:
    vector_rank = {query_id: ranking[:RRF_DEPTH] for query_id, ranking in numpy_vector_rank.items()}
published_vector_top5_exact_queries = sum(vector_rank[query_id][:5] == published_vector[query_id] for query_id in search_queries)
assert published_vector_top5_exact_queries == 30 and distance_matches_squared_l2
print({'normalized_bm25_top5_exact_queries': 30, 'cache_items': len(vectors_by_key), 'dimension': chunk_vectors.shape[1], 'dtype': str(chunk_vectors.dtype), 'finite': bool(np.isfinite(chunk_vectors).all()), 'numpy_l2_top5_exact_queries': numpy_published_top5_exact_queries, 'tmp_snapshot_used': vector_fallback_used, 'snapshot_top5_exact_queries': published_vector_top5_exact_queries, 'snapshot_attempt_top5_exact_queries': snapshot_attempt_top5_exact_queries, 'snapshot_attempt_baseline_rrf_exact_queries': snapshot_attempt_baseline_rrf_exact_queries})

{'normalized_bm25_top5_exact_queries': 30, 'cache_items': 357, 'dimension': 1536, 'dtype': 'float32', 'finite': True, 'numpy_l2_top5_exact_queries': 20, 'tmp_snapshot_used': True, 'snapshot_top5_exact_queries': 30, 'snapshot_attempt_top5_exact_queries': [30], 'snapshot_attempt_baseline_rrf_exact_queries': [30]}


## Weighted fusion, evaluation, candidate coverage, and automatic selection

각 query의 BM25 top-50과 Vector top-50 union만 fusion 후보로 사용한다. Ranking을 모두 생성한 다음에만 expected card/level/required terms/category를 strict relevance, metric, grouping, coverage annotation에 사용한다.

In [4]:
def weighted_rrf_details(bm25_ranking, vector_ranking, vector_weight, bm25_weight):
    bm25_rank = {identifier: rank for rank, identifier in enumerate(bm25_ranking[:RRF_DEPTH], 1)}
    vector_rank_lookup = {identifier: rank for rank, identifier in enumerate(vector_ranking[:RRF_DEPTH], 1)}
    union_ids = set(bm25_rank) | set(vector_rank_lookup)
    details = {}
    for identifier in union_ids:
        vector_score = vector_weight / (RRF_K + vector_rank_lookup[identifier]) if identifier in vector_rank_lookup else 0.0
        bm25_score = bm25_weight / (RRF_K + bm25_rank[identifier]) if identifier in bm25_rank else 0.0
        details[identifier] = {
            'vector_rank': vector_rank_lookup.get(identifier),
            'bm25_rank': bm25_rank.get(identifier),
            'vector_score': vector_score,
            'bm25_score': bm25_score,
            'total_score': vector_score + bm25_score,
        }
    ranking = sorted(union_ids, key=lambda identifier: (-details[identifier]['total_score'], identifier))
    return ranking, details


FUSED_RANKINGS, FUSED_DETAILS = {}, {}
for configuration, (vector_weight, bm25_weight) in CONFIGURATIONS.items():
    FUSED_RANKINGS[configuration], FUSED_DETAILS[configuration] = {}, {}
    for query_id in search_queries:
        ranking, details = weighted_rrf_details(normalized_bm25[query_id], vector_rank[query_id], vector_weight, bm25_weight)
        FUSED_RANKINGS[configuration][query_id] = ranking
        FUSED_DETAILS[configuration][query_id] = details
        assert set(ranking) == set(normalized_bm25[query_id][:50]) | set(vector_rank[query_id][:50])
        assert len(ranking) >= 50


def relevant_ids(evaluation):
    return {
        chunk['id'] for chunk in chunks
        if chunk['metadata']['card_key'] == evaluation['expected_card']
        and chunk['metadata']['level'] == evaluation['expected_level']
        and all(normalized_text(term) in normalized_text(chunk['document']) for term in evaluation['required_terms'])
    }


def metrics(evaluation, ranking):
    relevant = relevant_ids(evaluation)
    hits = [identifier in relevant for identifier in ranking[:5]]
    first = next((rank for rank, hit in enumerate(hits, 1) if hit), None)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, 1))
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    return {
        'card_hit_at_3': int(any(chunk_by_id[identifier]['metadata']['card_key'] == evaluation['expected_card'] for identifier in ranking[:3])),
        'strict_evidence_hit_at_3': int(any(hits[:3])),
        'recall_at_5': sum(hits) / len(relevant),
        'mrr_at_5': 1 / first if first else 0.0,
        'ndcg_at_5': dcg / ideal if ideal else 0.0,
    }


metric_names = ('card_hit_at_3', 'strict_evidence_hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5')
per_query_rows = []
for configuration, rankings in FUSED_RANKINGS.items():
    vector_weight, bm25_weight = CONFIGURATIONS[configuration]
    for query_id, evaluation in evaluation_by_id.items():
        ranking = rankings[query_id]
        per_query_rows.append({
            'configuration': configuration, 'vector_weight': vector_weight, 'bm25_weight': bm25_weight,
            'query_id': query_id, 'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence',
            'category': evaluation['category'], 'query': evaluation['query'],
            'expected_card': evaluation['expected_card'], 'expected_level': evaluation['expected_level'],
            **metrics(evaluation, ranking),
            'top5_chunk_ids': canonical_json(ranking[:5]),
            'top5_cards': canonical_json([chunk_by_id[identifier]['metadata']['card_key'] for identifier in ranking[:5]]),
            'top5_levels': canonical_json([chunk_by_id[identifier]['metadata']['level'] for identifier in ranking[:5]]),
        })


def aggregate(rows):
    return {**{metric: sum(row[metric] for row in rows) / len(rows) for metric in metric_names}, 'denominator': len(rows)}


summary_rows = []
for configuration in CONFIGURATIONS:
    selected = [row for row in per_query_rows if row['configuration'] == configuration]
    groups = {
        'all': selected,
        'card': [row for row in selected if row['question_group'] == 'card'],
        'evidence': [row for row in selected if row['question_group'] == 'evidence'],
        **{f'category_{category}': [row for row in selected if row['category'] == category] for category in ('proper_noun', 'numeric_condition', 'semantic')},
    }
    vector_weight, bm25_weight = CONFIGURATIONS[configuration]
    for group, rows in groups.items():
        summary_rows.append({'configuration': configuration, 'vector_weight': vector_weight, 'bm25_weight': bm25_weight, 'question_group': group, **aggregate(rows)})

assert len(per_query_rows) == 5 * 30 == 150 and len(summary_rows) == 5 * 6 == 30
per_query_map = {(row['configuration'], row['query_id']): row for row in per_query_rows}
summary_map = {(row['configuration'], row['question_group']): row for row in summary_rows}

# Baseline exact regression against the published 13 normalized RRF.
saved_baseline_rows = {row['query_id']: row for row in saved_rows if row['configuration'] == 'rrf_vector_normalized'}
for query_id in search_queries:
    current, previous = per_query_map[(BASELINE_CONFIGURATION, query_id)], saved_baseline_rows[query_id]
    for field in (*metric_names, 'top5_chunk_ids', 'top5_cards', 'top5_levels'):
        assert str(current[field]) == previous[field], (query_id, field, current[field], previous[field])
saved_baseline_summary = {row['question_group']: row for row in saved_summary['summaries'] if row['configuration'] == 'rrf_vector_normalized'}
for group in ('all', 'card', 'evidence', 'category_proper_noun', 'category_numeric_condition', 'category_semantic'):
    current = summary_map[(BASELINE_CONFIGURATION, group)]
    previous = saved_baseline_summary[group]
    assert {field: current[field] for field in (*metric_names, 'denominator')} == {field: previous[field] for field in (*metric_names, 'denominator')}

baseline_evidence_rows = {row['query_id']: row for row in per_query_rows if row['configuration'] == BASELINE_CONFIGURATION and row['question_group'] == 'evidence'}
paired_comparisons = {}
for configuration in CONFIGURATIONS:
    counts = {}
    guardrail_loss_queries = []
    for metric in ('mrr_at_5', 'ndcg_at_5'):
        deltas = [per_query_map[(configuration, query_id)][metric] - baseline_evidence_rows[query_id][metric] for query_id in baseline_evidence_rows]
        counts[metric] = {
            'wins': sum(delta > TOLERANCE for delta in deltas),
            'losses': sum(delta < -TOLERANCE for delta in deltas),
            'ties': sum(abs(delta) <= TOLERANCE for delta in deltas),
            'mean_delta': sum(deltas) / len(deltas),
        }
        assert counts[metric]['wins'] + counts[metric]['losses'] + counts[metric]['ties'] == 20
    for query_id, evaluation in evaluation_by_id.items():
        current = per_query_map[(configuration, query_id)]
        baseline = per_query_map[(BASELINE_CONFIGURATION, query_id)]
        fields = ('strict_evidence_hit_at_3', 'recall_at_5', 'ndcg_at_5', 'card_hit_at_3') if evaluation['expected_level'] != 'card' else ('card_hit_at_3',)
        losses = [field for field in fields if current[field] < baseline[field] - TOLERANCE]
        if losses:
            guardrail_loss_queries.append({'query_id': query_id, 'metrics': losses})
    paired_comparisons[configuration] = {**counts, 'guardrail_loss_queries': guardrail_loss_queries}
    evidence_summary = summary_map[(configuration, 'evidence')]
    evidence_summary.update({
        'paired_mrr_wins': counts['mrr_at_5']['wins'], 'paired_mrr_losses': counts['mrr_at_5']['losses'], 'paired_mrr_ties': counts['mrr_at_5']['ties'],
        'paired_ndcg_wins': counts['ndcg_at_5']['wins'], 'paired_ndcg_losses': counts['ndcg_at_5']['losses'], 'paired_ndcg_ties': counts['ndcg_at_5']['ties'],
        'guardrail_loss_query_count': len(guardrail_loss_queries),
    })

candidate_recall_rows = []
union_coverage_rows = []
candidate_rows = []
for query_id, evaluation in evaluation_by_id.items():
    relevant = relevant_ids(evaluation)
    bm25_top50, vector_top50 = normalized_bm25[query_id][:50], vector_rank[query_id][:50]
    union_ids = set(bm25_top50) | set(vector_top50)
    missing_relevant = sorted(relevant - union_ids)
    expected_card_ids = {chunk['id'] for chunk in chunks if chunk['metadata']['card_key'] == evaluation['expected_card']}
    union_coverage_rows.append({
        'query_id': query_id, 'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence', 'category': evaluation['category'],
        'bm25_top50_count': len(bm25_top50), 'vector_top50_count': len(vector_top50), 'union_unique_count': len(union_ids),
        'component_overlap_count': len(set(bm25_top50) & set(vector_top50)),
        'bm25_only_count': len(set(bm25_top50) - set(vector_top50)), 'vector_only_count': len(set(vector_top50) - set(bm25_top50)),
        'strict_relevant_count': len(relevant), 'union_strict_hit': int(bool(relevant & union_ids)),
        'union_strict_recall': len(relevant & union_ids) / len(relevant),
        'union_expected_card_hit': int(bool(expected_card_ids & union_ids)),
        'union_expected_card_candidate_count': len(expected_card_ids & union_ids),
        'missing_relevant_ids': canonical_json(missing_relevant),
    })
    assert set(bm25_top50).issubset(union_ids) and set(vector_top50).issubset(union_ids)
    assert 50 <= len(union_ids) <= 100
    for configuration, ranking_by_query in FUSED_RANKINGS.items():
        vector_weight, bm25_weight = CONFIGURATIONS[configuration]
        ranking = ranking_by_query[query_id]
        row = {
            'configuration': configuration, 'vector_weight': vector_weight, 'bm25_weight': bm25_weight,
            'query_id': query_id, 'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence',
            'category': evaluation['category'], 'strict_relevant_count': len(relevant), 'union_unique_count': len(union_ids),
        }
        previous_ids, previous_recall = set(), -1.0
        for depth in CANDIDATE_DEPTHS:
            candidate_ids = set(ranking[:depth])
            recall = len(candidate_ids & relevant) / len(relevant)
            row.update({
                f'candidate_count_at_{depth}': len(ranking[:depth]),
                f'strict_hit_at_{depth}': int(bool(candidate_ids & relevant)),
                f'strict_recall_at_{depth}': recall,
                f'expected_card_hit_at_{depth}': int(any(chunk_by_id[identifier]['metadata']['card_key'] == evaluation['expected_card'] for identifier in ranking[:depth])),
            })
            assert previous_ids.issubset(candidate_ids) and recall + TOLERANCE >= previous_recall
            previous_ids, previous_recall = candidate_ids, recall
        candidate_recall_rows.append(row)
        details = FUSED_DETAILS[configuration][query_id]
        for fused_rank, identifier in enumerate(ranking[:50], 1):
            detail = details[identifier]
            candidate_rows.append({
                'configuration': configuration, 'vector_weight': vector_weight, 'bm25_weight': bm25_weight,
                'query_id': query_id, 'fused_rank': fused_rank, 'chunk_id': identifier,
                'vector_rank': detail['vector_rank'] if detail['vector_rank'] is not None else '',
                'bm25_rank': detail['bm25_rank'] if detail['bm25_rank'] is not None else '',
                'vector_score': detail['vector_score'], 'bm25_score': detail['bm25_score'], 'total_score': detail['total_score'],
                'strict_relevant': int(identifier in relevant),
                'expected_card': int(chunk_by_id[identifier]['metadata']['card_key'] == evaluation['expected_card']),
                'card_key': chunk_by_id[identifier]['metadata']['card_key'], 'level': chunk_by_id[identifier]['metadata']['level'],
            })

assert len(candidate_recall_rows) == 5 * 30 == 150
assert len(union_coverage_rows) == 30
assert len(candidate_rows) == 5 * 30 * 50 == 7500
for row in candidate_recall_rows:
    assert row['candidate_count_at_10'] <= row['candidate_count_at_20'] <= row['candidate_count_at_50']
    assert row['strict_hit_at_10'] <= row['strict_hit_at_20'] <= row['strict_hit_at_50']
    assert row['strict_recall_at_10'] <= row['strict_recall_at_20'] + TOLERANCE <= row['strict_recall_at_50'] + 2 * TOLERANCE
    assert row['expected_card_hit_at_10'] <= row['expected_card_hit_at_20'] <= row['expected_card_hit_at_50']
    assert all(0 <= row[field] <= 1 for field in row if field.startswith(('strict_hit_', 'strict_recall_', 'expected_card_hit_')))
assert all(0 <= row[metric] <= 1 for row in per_query_rows for metric in metric_names)
assert all(0 <= row[metric] <= 1 for row in summary_rows for metric in metric_names)

baseline_evidence = summary_map[(BASELINE_CONFIGURATION, 'evidence')]
baseline_card = summary_map[(BASELINE_CONFIGURATION, 'card')]
qualification = {}
for configuration, (vector_weight, bm25_weight) in CONFIGURATIONS.items():
    evidence, card = summary_map[(configuration, 'evidence')], summary_map[(configuration, 'card')]
    delta = evidence['mrr_at_5'] - baseline_evidence['mrr_at_5']
    guardrails = {
        'evidence_strict_evidence_hit_at_3': evidence['strict_evidence_hit_at_3'] >= baseline_evidence['strict_evidence_hit_at_3'] - TOLERANCE,
        'evidence_recall_at_5': evidence['recall_at_5'] >= baseline_evidence['recall_at_5'] - TOLERANCE,
        'evidence_ndcg_at_5': evidence['ndcg_at_5'] >= baseline_evidence['ndcg_at_5'] - TOLERANCE,
        'evidence_card_hit_at_3': evidence['card_hit_at_3'] >= baseline_evidence['card_hit_at_3'] - TOLERANCE,
        'card_card_hit_at_3': card['card_hit_at_3'] >= baseline_card['card_hit_at_3'] - TOLERANCE,
    }
    qualification[configuration] = {'vector_weight': vector_weight, 'bm25_weight': bm25_weight, 'evidence_mrr_delta': delta, 'primary_pass': delta >= PRIMARY_DELTA, 'guardrails': guardrails, 'qualified': delta >= PRIMARY_DELTA and all(guardrails.values())}


def compare_qualified(left, right):
    left_summary, right_summary = summary_map[(left, 'evidence')], summary_map[(right, 'evidence')]
    for metric in ('mrr_at_5', 'ndcg_at_5', 'recall_at_5', 'strict_evidence_hit_at_3', 'card_hit_at_3'):
        difference = left_summary[metric] - right_summary[metric]
        if abs(difference) > TOLERANCE:
            return -1 if difference > 0 else 1
    left_distance = abs(CONFIGURATIONS[left][0] - 0.5)
    right_distance = abs(CONFIGURATIONS[right][0] - 0.5)
    if abs(left_distance - right_distance) > TOLERANCE:
        return -1 if left_distance < right_distance else 1
    difference = CONFIGURATIONS[left][0] - CONFIGURATIONS[right][0]
    if abs(difference) > TOLERANCE:
        return -1 if difference > 0 else 1
    return -1 if left < right else (1 if left > right else 0)


qualified_configurations = [configuration for configuration in CONFIGURATIONS if qualification[configuration]['qualified']]
selected_configuration = sorted(qualified_configurations, key=cmp_to_key(compare_qualified))[0] if qualified_configurations else BASELINE_CONFIGURATION
selection_decision = 'select_qualified_weight' if qualified_configurations else 'retain_baseline_0.5_0.5'

candidate_recall_aggregate = []
for configuration in CONFIGURATIONS:
    rows = [row for row in candidate_recall_rows if row['configuration'] == configuration]
    for depth in CANDIDATE_DEPTHS:
        candidate_recall_aggregate.append({
            'configuration': configuration, 'depth': depth, 'queries': len(rows),
            'mean_candidate_count': sum(row[f'candidate_count_at_{depth}'] for row in rows) / len(rows),
            'strict_hit_rate': sum(row[f'strict_hit_at_{depth}'] for row in rows) / len(rows),
            'mean_strict_recall': sum(row[f'strict_recall_at_{depth}'] for row in rows) / len(rows),
            'expected_card_hit_rate': sum(row[f'expected_card_hit_at_{depth}'] for row in rows) / len(rows),
        })

atomic_csv(OUTPUT_ROOT / 'rrf_weight_ablation_per_query.csv', per_query_rows)
atomic_csv(OUTPUT_ROOT / 'rrf_weight_ablation_summary.csv', summary_rows)
atomic_csv(OUTPUT_ROOT / 'rrf_weight_candidate_recall.csv', candidate_recall_rows)
atomic_csv(OUTPUT_ROOT / 'rrf_weight_union_coverage.csv', union_coverage_rows)
atomic_csv(OUTPUT_ROOT / 'rrf_weight_candidates.csv', candidate_rows)

readme = f'''# Normalized RRF weight ablation

이 디렉터리는 기존 개발 질의 30개로 Vector:Normalized BM25 RRF 가중치 5개를 비교한 결과다. 독립 holdout은 사용하지 않았으며 운영 또는 미관측 데이터 일반화를 주장하지 않는다.

## Reproduction

- 환경: `conda run -n skn25`
- 실행: `nbclient 0.10.4`, `NotebookClient(timeout=900, kernel_name="python3")`
- 노트북: `notebooks/16_normalized_rrf_weight_ablation.ipynb`
- network/API/new embeddings/package installs: 0

## Automatic selection

- decision: `{selection_decision}`
- selected configuration: `{selected_configuration}`
- qualified alternatives: `{', '.join(qualified_configurations) if qualified_configurations else 'none'}`
- 사전 자격·tie-break 규칙으로 선택한 개발셋 후보일 뿐 일반적 우월성 결론이 아니다.
'''
(OUTPUT_ROOT / 'README.md').write_text(readme, encoding='utf-8')

output_hashes = {name: sha256_file(OUTPUT_ROOT / name) for name in (
    'rrf_weight_ablation_per_query.csv', 'rrf_weight_ablation_summary.csv', 'rrf_weight_candidate_recall.csv',
    'rrf_weight_union_coverage.csv', 'rrf_weight_candidates.csv', 'README.md',
)}
summary_result = {
    'schema_version': 'normalized_rrf_weight_ablation_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'scope': {'dataset': 'development_queries_only', 'queries': 30, 'evidence_queries': 20, 'card_queries': 10, 'independent_holdout_used': False, 'operational_generalization_claimed': False},
    'contract': {'configurations': CONFIGURATIONS, 'bm25': {'k1': BM25_K1, 'b': BM25_B}, 'rrf': {'k': RRF_K, 'depth': RRF_DEPTH, 'formula': 'vector_weight/(60+vector_rank)+bm25_weight/(60+bm25_rank)', 'tie': 'raw total descending then chunk_id lexical; no rounding'}, 'selection': SELECTION_CONTRACT},
    'results': {'summaries': summary_rows, 'paired_evidence': paired_comparisons, 'candidate_recall_aggregate': candidate_recall_aggregate, 'union_coverage': {'queries': len(union_coverage_rows), 'mean_unique_count': sum(row['union_unique_count'] for row in union_coverage_rows) / len(union_coverage_rows), 'strict_hit_rate': sum(row['union_strict_hit'] for row in union_coverage_rows) / len(union_coverage_rows), 'mean_strict_recall': sum(row['union_strict_recall'] for row in union_coverage_rows) / len(union_coverage_rows), 'expected_card_hit_rate': sum(row['union_expected_card_hit'] for row in union_coverage_rows) / len(union_coverage_rows), 'queries_with_missing_relevant_ids': [row['query_id'] for row in union_coverage_rows if json.loads(row['missing_relevant_ids'])]}},
    'selection': {'baseline': BASELINE_CONFIGURATION, 'qualification': qualification, 'qualified_configurations': qualified_configurations, 'selected_configuration': selected_configuration, 'decision': selection_decision},
    'baseline_reproduction': {'normalized_bm25_top5_exact_queries': 30, 'vector_top5_exact_queries': published_vector_top5_exact_queries, 'baseline_rrf_top5_per_query_summary_exact': True},
    'vector_reproduction': {'embedding_model': EMBEDDING_MODEL, 'cached_items': len(vectors_by_key), 'chunks': len(chunks), 'queries': len(search_queries), 'dimension': 1536, 'dtype': str(chunk_vectors.dtype), 'finite': bool(np.isfinite(chunk_vectors).all()), 'cache_validation': cache_validation, 'sqlite_read_only': True, 'sqlite_collection_config': collection_config, 'sqlite_distance_override_rows': distance_override_rows, 'numpy_bruteforce_attempted_first': True, 'numpy_published_top5_exact_queries': numpy_published_top5_exact_queries, 'fallback_to_tmp_byte_snapshot': vector_fallback_used, 'snapshot_published_top5_exact_queries': published_vector_top5_exact_queries, 'snapshot_attempt_top5_exact_queries': snapshot_attempt_top5_exact_queries, 'snapshot_attempt_baseline_rrf_exact_queries': snapshot_attempt_baseline_rrf_exact_queries, 'snapshot_attempts_used': len(snapshot_attempt_top5_exact_queries), 'snapshot_acceptance_contract': 'first temporary snapshot attempt with both published vector top-5 and baseline 0.5:0.5 RRF top-5 exact 30/30; maximum 8 attempts', 'snapshot_distances_match_squared_l2': bool(distance_matches_squared_l2)},
    'execution': {'environment': 'skn25', 'runner': 'nbclient 0.10.4 NotebookClient', 'network_calls': 0, 'api_calls': 0, 'new_embeddings': 0, 'package_install_calls': 0},
    'integrity': {'input_hashes_before': input_hashes_before, 'chroma_tree_hash_before': chroma_tree_hash_before, 'chroma_count_before': chroma_count_before, 'output_hashes': output_hashes},
    'limitations': ['Only 30 development queries were used; no independent generalization estimate is available.', 'All configurations reuse structured-assisted chunk boundaries inherited from the source artifacts.', 'NumPy exact L2 does not reproduce every approximate Chroma top-5, so the verified temporary snapshot supplies vector depth-50 ranks.', 'Only five predeclared weights and depth-50 RRF components were evaluated.', 'Coverage metrics depend on the existing strict relevance contract and incomplete source coverage limitations.'],
}
atomic_json(OUTPUT_ROOT / 'rrf_weight_ablation_summary.json', summary_result)

manifest_files = {
    **{f'input:{name}': {'path': path.relative_to(PROJECT_ROOT).as_posix(), 'sha256': input_hashes_before[name]} for name, path in ALL_INPUT_FILES.items()},
    **{f'output:{name}': {'path': f'notebooks/data/16_normalized_rrf_weight_ablation/{name}', 'sha256': sha256_file(OUTPUT_ROOT / name)} for name in ('rrf_weight_ablation_per_query.csv', 'rrf_weight_ablation_summary.csv', 'rrf_weight_ablation_summary.json', 'rrf_weight_candidate_recall.csv', 'rrf_weight_union_coverage.csv', 'rrf_weight_candidates.csv', 'README.md')},
    'output:notebook': {'path': NOTEBOOK_PATH.relative_to(PROJECT_ROOT).as_posix(), 'sha256': 'pending_after_nbclient_serialization'},
}
atomic_json(OUTPUT_ROOT / 'rrf_weight_run_manifest.json', {'schema_version': 'normalized_rrf_weight_run_manifest_v1', 'self_hash_excluded': True, 'files': manifest_files})
print({'evidence_summaries': {configuration: summary_map[(configuration, 'evidence')] for configuration in CONFIGURATIONS}, 'qualification': qualification, 'decision': selection_decision, 'selected': selected_configuration, 'candidate_rows': len(candidate_rows), 'union_mean_unique': summary_result['results']['union_coverage']['mean_unique_count']})

{'evidence_summaries': {'vector_0.4_bm25_0.6': {'configuration': 'vector_0.4_bm25_0.6', 'vector_weight': 0.4, 'bm25_weight': 0.6, 'question_group': 'evidence', 'card_hit_at_3': 0.9, 'strict_evidence_hit_at_3': 0.75, 'recall_at_5': 0.7125, 'mrr_at_5': 0.6199999999999999, 'ndcg_at_5': 0.6128787801395787, 'denominator': 20, 'paired_mrr_wins': 2, 'paired_mrr_losses': 1, 'paired_mrr_ties': 17, 'paired_ndcg_wins': 2, 'paired_ndcg_losses': 2, 'paired_ndcg_ties': 16, 'guardrail_loss_query_count': 2}, 'vector_0.5_bm25_0.5': {'configuration': 'vector_0.5_bm25_0.5', 'vector_weight': 0.5, 'bm25_weight': 0.5, 'question_group': 'evidence', 'card_hit_at_3': 0.9, 'strict_evidence_hit_at_3': 0.75, 'recall_at_5': 0.7125, 'mrr_at_5': 0.6033333333333333, 'ndcg_at_5': 0.5965505415086586, 'denominator': 20, 'paired_mrr_wins': 0, 'paired_mrr_losses': 0, 'paired_mrr_ties': 20, 'paired_ndcg_wins': 0, 'paired_ndcg_losses': 0, 'paired_ndcg_ties': 20, 'guardrail_loss_query_count': 0}, 'vector_0.6_bm25_0.4': {'con

## Final integrity and schema checks

출력 CSV/JSON을 다시 읽어 row/schema/range를 검사하고, 허용 입력·cache·Chroma raw hash/tree hash/count가 실행 전후 동일함을 확인한다. Manifest는 self hash를 제외하고 실행 드라이버가 최종 notebook hash를 채운다.

동일 file-hash map에서 tree digest를 재구성하고 input hash 직후, snapshot copy 전후, query 후, 마지막 산출물 저장 후 원본 불변성을 반복 확인한다.

In [5]:
integrity_stage_checks.append(assert_current_chroma(chroma_file_hashes_before, FROZEN_EXPECTED_CHROMA_TREE_HASH, 'after_initial_artifact_save'))
input_hashes_after = {name: sha256_file(path) for name, path in ALL_INPUT_FILES.items()}
chroma_file_hashes_after = file_hash_map(CHROMA_ROOT)
chroma_tree_hash_after = tree_hash_from_file_hash_map(chroma_file_hashes_after)
assert chroma_tree_hash_after == tree_hash(CHROMA_ROOT)
assert chroma_file_hashes_after == chroma_file_hashes_before
with sqlite3.connect(f'file:{database}?mode=ro', uri=True) as connection:
    chroma_count_after = connection.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]
assert input_hashes_after == input_hashes_before
assert chroma_tree_hash_after == chroma_tree_hash_before
assert chroma_count_after == chroma_count_before == 327

final_summary = json.loads((OUTPUT_ROOT / 'rrf_weight_ablation_summary.json').read_text(encoding='utf-8'))
final_summary['integrity'].update({
    'input_hashes_after': input_hashes_after,
    'chroma_file_hashes_before': chroma_file_hashes_before,
    'chroma_file_hashes_after': chroma_file_hashes_after,
    'chroma_tree_hash_before': chroma_tree_hash_before,
    'chroma_tree_hash_after': chroma_tree_hash_after,
    'chroma_tree_hash_algorithm': 'deterministic directory preorder (root files first, sorted names/subdirectories) + relative path bytes + raw SHA-256 digest bytes',
    'chroma_tree_hash_reconstructed_from_file_hash_map_exact': True,
    'frozen_expected_chroma_tree_hash_from_15': FROZEN_EXPECTED_CHROMA_TREE_HASH,
    'chroma_count_after': chroma_count_after,
    'inputs_cache_chroma_unchanged': True,
    'integrity_check_stages': integrity_stage_checks,
})
atomic_json(OUTPUT_ROOT / 'rrf_weight_ablation_summary.json', final_summary)
final_manifest = json.loads((OUTPUT_ROOT / 'rrf_weight_run_manifest.json').read_text(encoding='utf-8'))
final_manifest['files']['output:rrf_weight_ablation_summary.json']['sha256'] = sha256_file(OUTPUT_ROOT / 'rrf_weight_ablation_summary.json')
atomic_json(OUTPUT_ROOT / 'rrf_weight_run_manifest.json', final_manifest)

stored_per_query = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_ablation_per_query.csv').open(encoding='utf-8')))
stored_summary = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_ablation_summary.csv').open(encoding='utf-8')))
stored_candidate_recall = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_candidate_recall.csv').open(encoding='utf-8')))
stored_union = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_union_coverage.csv').open(encoding='utf-8')))
stored_candidates = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_candidates.csv').open(encoding='utf-8')))
stored_json = json.loads((OUTPUT_ROOT / 'rrf_weight_ablation_summary.json').read_text(encoding='utf-8'))
assert (len(stored_per_query), len(stored_summary), len(stored_candidate_recall), len(stored_union), len(stored_candidates)) == (150, 30, 150, 30, 7500)
assert set(row['configuration'] for row in stored_per_query) == set(CONFIGURATIONS)
assert all(sum(row['configuration'] == configuration for row in stored_per_query) == 30 for configuration in CONFIGURATIONS)
assert stored_json['baseline_reproduction']['baseline_rrf_top5_per_query_summary_exact']
assert stored_json['vector_reproduction']['snapshot_published_top5_exact_queries'] == 30
assert stored_json['execution']['network_calls'] == stored_json['execution']['api_calls'] == stored_json['execution']['new_embeddings'] == 0
assert stored_json['selection']['selected_configuration'] in CONFIGURATIONS
assert stored_json['integrity']['inputs_cache_chroma_unchanged']
assert json.loads((OUTPUT_ROOT / 'rrf_weight_run_manifest.json').read_text(encoding='utf-8'))['self_hash_excluded']
print({'code_cells_compile': 'validated externally and all cells executed', 'per_query_rows': len(stored_per_query), 'summary_rows': len(stored_summary), 'candidate_recall_rows': len(stored_candidate_recall), 'union_rows': len(stored_union), 'candidate_rows': len(stored_candidates), 'baseline_exact': True, 'source_unchanged': True, 'network_api_new_embeddings': 0, 'selection': stored_json['selection']['decision']})

{'code_cells_compile': 'validated externally and all cells executed', 'per_query_rows': 150, 'summary_rows': 30, 'candidate_recall_rows': 150, 'union_rows': 30, 'candidate_rows': 7500, 'baseline_exact': True, 'source_unchanged': True, 'network_api_new_embeddings': 0, 'selection': 'retain_baseline_0.5_0.5'}


## Adaptive BM25-heavy exploratory follow-up

이 섹션의 `Vector:Normalized BM25 = 0.3:0.7`, `0.2:0.8`은 위에서 사전 고정한 최초 5개 결과 중 `0.4:0.6`의 evidence MRR 개선을 확인한 **뒤에 사용자가 요청한 adaptive exploratory test**다. 따라서 최초 실험의 자동판정 `retain_baseline_0.5_0.5`을 소급 변경하지 않는다.

새 두 가중치도 BM25 `k1=1.5, b=0.75`, RRF `k=60, depth=50`, raw fused score 정렬과 chunk ID lexical tie-break를 그대로 사용한다. 참고 gate는 최초 baseline `0.5:0.5` 대비 evidence MRR delta `>= +0.025`와 기존 전체 guardrail 비회귀다. 통과하더라도 `exploratory_candidate_for_confirmatory_retest`로만 기록한다. 전체 7개 비교와 최초 자동판정, 사후 참고 판정은 summary의 별도 필드에 저장한다.

In [6]:
# Adaptive follow-up: preserve the initial five configurations, then append two BM25-heavy weights.
INITIAL_CONFIGURATIONS = dict(CONFIGURATIONS)
INITIAL_SELECTION = json.loads(canonical_json(summary_result['selection']))
INITIAL_OUTPUT_HASHES = dict(summary_result['integrity']['output_hashes'])
INITIAL_PER_QUERY_ROWS = [dict(row) for row in per_query_rows]
INITIAL_SUMMARY_ROWS = [dict(row) for row in summary_rows]
INITIAL_CANDIDATE_RECALL_ROWS = [dict(row) for row in candidate_recall_rows]
INITIAL_CANDIDATE_ROWS = [dict(row) for row in candidate_rows]

FOLLOWUP_WEIGHT_PAIRS = ((0.3, 0.7), (0.2, 0.8))
FOLLOWUP_CONFIGURATIONS = {
    configuration_name(vector_weight, bm25_weight): (vector_weight, bm25_weight)
    for vector_weight, bm25_weight in FOLLOWUP_WEIGHT_PAIRS
}
assert set(FOLLOWUP_CONFIGURATIONS).isdisjoint(INITIAL_CONFIGURATIONS)
CONFIGURATIONS = {**INITIAL_CONFIGURATIONS, **FOLLOWUP_CONFIGURATIONS}

for configuration, (vector_weight, bm25_weight) in FOLLOWUP_CONFIGURATIONS.items():
    FUSED_RANKINGS[configuration], FUSED_DETAILS[configuration] = {}, {}
    for query_id in search_queries:
        ranking, details = weighted_rrf_details(normalized_bm25[query_id], vector_rank[query_id], vector_weight, bm25_weight)
        FUSED_RANKINGS[configuration][query_id] = ranking
        FUSED_DETAILS[configuration][query_id] = details
        union_ids = set(normalized_bm25[query_id][:50]) | set(vector_rank[query_id][:50])
        assert set(ranking) == union_ids and len(ranking) >= 50

for configuration in FOLLOWUP_CONFIGURATIONS:
    vector_weight, bm25_weight = CONFIGURATIONS[configuration]
    for query_id, evaluation in evaluation_by_id.items():
        ranking = FUSED_RANKINGS[configuration][query_id]
        per_query_rows.append({
            'configuration': configuration, 'vector_weight': vector_weight, 'bm25_weight': bm25_weight,
            'query_id': query_id, 'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence',
            'category': evaluation['category'], 'query': evaluation['query'],
            'expected_card': evaluation['expected_card'], 'expected_level': evaluation['expected_level'],
            **metrics(evaluation, ranking),
            'top5_chunk_ids': canonical_json(ranking[:5]),
            'top5_cards': canonical_json([chunk_by_id[identifier]['metadata']['card_key'] for identifier in ranking[:5]]),
            'top5_levels': canonical_json([chunk_by_id[identifier]['metadata']['level'] for identifier in ranking[:5]]),
        })

for configuration in FOLLOWUP_CONFIGURATIONS:
    selected = [row for row in per_query_rows if row['configuration'] == configuration]
    groups = {
        'all': selected,
        'card': [row for row in selected if row['question_group'] == 'card'],
        'evidence': [row for row in selected if row['question_group'] == 'evidence'],
        **{f'category_{category}': [row for row in selected if row['category'] == category] for category in ('proper_noun', 'numeric_condition', 'semantic')},
    }
    vector_weight, bm25_weight = CONFIGURATIONS[configuration]
    for group, rows in groups.items():
        summary_rows.append({'configuration': configuration, 'vector_weight': vector_weight, 'bm25_weight': bm25_weight, 'question_group': group, **aggregate(rows)})

per_query_map = {(row['configuration'], row['query_id']): row for row in per_query_rows}
summary_map = {(row['configuration'], row['question_group']): row for row in summary_rows}
baseline_evidence_rows = {row['query_id']: row for row in per_query_rows if row['configuration'] == BASELINE_CONFIGURATION and row['question_group'] == 'evidence'}

for configuration in FOLLOWUP_CONFIGURATIONS:
    counts = {}
    guardrail_loss_queries = []
    for metric in ('mrr_at_5', 'ndcg_at_5'):
        deltas = [per_query_map[(configuration, query_id)][metric] - baseline_evidence_rows[query_id][metric] for query_id in baseline_evidence_rows]
        counts[metric] = {
            'wins': sum(delta > TOLERANCE for delta in deltas),
            'losses': sum(delta < -TOLERANCE for delta in deltas),
            'ties': sum(abs(delta) <= TOLERANCE for delta in deltas),
            'mean_delta': sum(deltas) / len(deltas),
        }
        assert counts[metric]['wins'] + counts[metric]['losses'] + counts[metric]['ties'] == 20
    for query_id, evaluation in evaluation_by_id.items():
        current = per_query_map[(configuration, query_id)]
        baseline = per_query_map[(BASELINE_CONFIGURATION, query_id)]
        fields = ('strict_evidence_hit_at_3', 'recall_at_5', 'ndcg_at_5', 'card_hit_at_3') if evaluation['expected_level'] != 'card' else ('card_hit_at_3',)
        losses = [field for field in fields if current[field] < baseline[field] - TOLERANCE]
        if losses:
            guardrail_loss_queries.append({'query_id': query_id, 'metrics': losses})
    paired_comparisons[configuration] = {**counts, 'guardrail_loss_queries': guardrail_loss_queries}
    evidence_summary = summary_map[(configuration, 'evidence')]
    evidence_summary.update({
        'paired_mrr_wins': counts['mrr_at_5']['wins'], 'paired_mrr_losses': counts['mrr_at_5']['losses'], 'paired_mrr_ties': counts['mrr_at_5']['ties'],
        'paired_ndcg_wins': counts['ndcg_at_5']['wins'], 'paired_ndcg_losses': counts['ndcg_at_5']['losses'], 'paired_ndcg_ties': counts['ndcg_at_5']['ties'],
        'guardrail_loss_query_count': len(guardrail_loss_queries),
    })

# Reuse and exactly assert the weight-independent union rows already persisted by the initial experiment.
stored_union_reference = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_union_coverage.csv').open(encoding='utf-8')))
assert len(stored_union_reference) == len(union_coverage_rows) == 30
for current, stored in zip(union_coverage_rows, stored_union_reference):
    assert current.keys() == stored.keys()
    for field in current:
        assert str(current[field]) == stored[field], (current['query_id'], field, current[field], stored[field])

for query_id, evaluation in evaluation_by_id.items():
    relevant = relevant_ids(evaluation)
    union_ids = set(normalized_bm25[query_id][:50]) | set(vector_rank[query_id][:50])
    for configuration in FOLLOWUP_CONFIGURATIONS:
        vector_weight, bm25_weight = CONFIGURATIONS[configuration]
        ranking = FUSED_RANKINGS[configuration][query_id]
        row = {
            'configuration': configuration, 'vector_weight': vector_weight, 'bm25_weight': bm25_weight,
            'query_id': query_id, 'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence',
            'category': evaluation['category'], 'strict_relevant_count': len(relevant), 'union_unique_count': len(union_ids),
        }
        previous_ids, previous_recall = set(), -1.0
        for depth in CANDIDATE_DEPTHS:
            candidate_ids = set(ranking[:depth])
            recall = len(candidate_ids & relevant) / len(relevant)
            row.update({
                f'candidate_count_at_{depth}': len(ranking[:depth]),
                f'strict_hit_at_{depth}': int(bool(candidate_ids & relevant)),
                f'strict_recall_at_{depth}': recall,
                f'expected_card_hit_at_{depth}': int(any(chunk_by_id[identifier]['metadata']['card_key'] == evaluation['expected_card'] for identifier in ranking[:depth])),
            })
            assert previous_ids.issubset(candidate_ids) and recall + TOLERANCE >= previous_recall
            previous_ids, previous_recall = candidate_ids, recall
        candidate_recall_rows.append(row)
        details = FUSED_DETAILS[configuration][query_id]
        for fused_rank, identifier in enumerate(ranking[:50], 1):
            detail = details[identifier]
            candidate_rows.append({
                'configuration': configuration, 'vector_weight': vector_weight, 'bm25_weight': bm25_weight,
                'query_id': query_id, 'fused_rank': fused_rank, 'chunk_id': identifier,
                'vector_rank': detail['vector_rank'] if detail['vector_rank'] is not None else '',
                'bm25_rank': detail['bm25_rank'] if detail['bm25_rank'] is not None else '',
                'vector_score': detail['vector_score'], 'bm25_score': detail['bm25_score'], 'total_score': detail['total_score'],
                'strict_relevant': int(identifier in relevant),
                'expected_card': int(chunk_by_id[identifier]['metadata']['card_key'] == evaluation['expected_card']),
                'card_key': chunk_by_id[identifier]['metadata']['card_key'], 'level': chunk_by_id[identifier]['metadata']['level'],
            })

for configuration in FOLLOWUP_CONFIGURATIONS:
    rows = [row for row in candidate_recall_rows if row['configuration'] == configuration]
    for depth in CANDIDATE_DEPTHS:
        candidate_recall_aggregate.append({
            'configuration': configuration, 'depth': depth, 'queries': len(rows),
            'mean_candidate_count': sum(row[f'candidate_count_at_{depth}'] for row in rows) / len(rows),
            'strict_hit_rate': sum(row[f'strict_hit_at_{depth}'] for row in rows) / len(rows),
            'mean_strict_recall': sum(row[f'strict_recall_at_{depth}'] for row in rows) / len(rows),
            'expected_card_hit_rate': sum(row[f'expected_card_hit_at_{depth}'] for row in rows) / len(rows),
        })

# The original five rows are byte-value equivalent in memory after appending the follow-up rows.
assert [row for row in per_query_rows if row['configuration'] in INITIAL_CONFIGURATIONS] == INITIAL_PER_QUERY_ROWS
assert [row for row in summary_rows if row['configuration'] in INITIAL_CONFIGURATIONS] == INITIAL_SUMMARY_ROWS
assert [row for row in candidate_recall_rows if row['configuration'] in INITIAL_CONFIGURATIONS] == INITIAL_CANDIDATE_RECALL_ROWS
assert [row for row in candidate_rows if row['configuration'] in INITIAL_CONFIGURATIONS] == INITIAL_CANDIDATE_ROWS
assert (len(per_query_rows), len(summary_rows), len(candidate_recall_rows), len(candidate_rows)) == (7 * 30, 7 * 6, 7 * 30, 7 * 30 * 50)

baseline_evidence = summary_map[(BASELINE_CONFIGURATION, 'evidence')]
baseline_card = summary_map[(BASELINE_CONFIGURATION, 'card')]
extended_qualification = {}
for configuration, (vector_weight, bm25_weight) in CONFIGURATIONS.items():
    evidence, card = summary_map[(configuration, 'evidence')], summary_map[(configuration, 'card')]
    delta = evidence['mrr_at_5'] - baseline_evidence['mrr_at_5']
    guardrails = {
        'evidence_strict_evidence_hit_at_3': evidence['strict_evidence_hit_at_3'] >= baseline_evidence['strict_evidence_hit_at_3'] - TOLERANCE,
        'evidence_recall_at_5': evidence['recall_at_5'] >= baseline_evidence['recall_at_5'] - TOLERANCE,
        'evidence_ndcg_at_5': evidence['ndcg_at_5'] >= baseline_evidence['ndcg_at_5'] - TOLERANCE,
        'evidence_card_hit_at_3': evidence['card_hit_at_3'] >= baseline_evidence['card_hit_at_3'] - TOLERANCE,
        'card_card_hit_at_3': card['card_hit_at_3'] >= baseline_card['card_hit_at_3'] - TOLERANCE,
    }
    extended_qualification[configuration] = {
        'vector_weight': vector_weight, 'bm25_weight': bm25_weight, 'evidence_mrr_delta': delta,
        'primary_pass': delta >= PRIMARY_DELTA, 'guardrails': guardrails,
        'qualified': delta >= PRIMARY_DELTA and all(guardrails.values()),
    }

extended_best_observed = sorted(CONFIGURATIONS, key=cmp_to_key(compare_qualified))[0]
followup_qualified = [configuration for configuration in FOLLOWUP_CONFIGURATIONS if extended_qualification[configuration]['qualified']]
followup_decision = 'exploratory_candidate_for_confirmatory_retest' if followup_qualified else 'no_exploratory_candidate_for_confirmatory_retest'

atomic_csv(OUTPUT_ROOT / 'rrf_weight_ablation_per_query.csv', per_query_rows)
atomic_csv(OUTPUT_ROOT / 'rrf_weight_ablation_summary.csv', summary_rows)
atomic_csv(OUTPUT_ROOT / 'rrf_weight_candidate_recall.csv', candidate_recall_rows)
atomic_csv(OUTPUT_ROOT / 'rrf_weight_candidates.csv', candidate_rows)

readme = f'''# Normalized RRF weight ablation

이 디렉터리는 기존 개발 질의 30개로 Vector:Normalized BM25 RRF 가중치를 비교한 결과다. 독립 holdout은 사용하지 않았으며 운영 또는 미관측 데이터 일반화를 주장하지 않는다.

## Experiment phases

- Initial predeclared weights: `0.4:0.6`, `0.5:0.5`, `0.6:0.4`, `0.7:0.3`, `0.8:0.2`
- Initial automatic decision: `{INITIAL_SELECTION['decision']}` / `{INITIAL_SELECTION['selected_configuration']}`
- Adaptive exploratory follow-up requested after observing the `0.4:0.6` result: `0.3:0.7`, `0.2:0.8`
- Follow-up reference decision: `{followup_decision}`
- Extended best observed by the fixed comparison order: `{extended_best_observed}`
- The adaptive follow-up does not retroactively alter the initial automatic decision.

## Reproduction

- 환경: `conda run -n skn25`
- 실행: `nbclient 0.10.4`, `NotebookClient(timeout=900, kernel_name="python3")`
- 노트북: `notebooks/16_normalized_rrf_weight_ablation.ipynb`
- network/API/new embeddings/package installs: 0
- Chroma integrity: 15번 실행 후 frozen tree hash를 시작 전에 검증하고, 동일 file-hash map에서 tree digest를 재구성하며 snapshot 전후와 마지막 저장 후 원본 불변성을 확인한다.
- Fresh-run comparison: 이전 7개 성능·candidate raw hash와 현재 fresh-run hash를 파일별 exact 비교하고, depth-50 차이가 있으면 summary에 원인과 새 결과를 기록한다.
'''
(OUTPUT_ROOT / 'README.md').write_text(readme, encoding='utf-8')

summary_result = json.loads((OUTPUT_ROOT / 'rrf_weight_ablation_summary.json').read_text(encoding='utf-8'))
summary_result['execution']['kernel_scope'] = 'fresh kernel; entire notebook executed from first cell through adaptive validation'
summary_result['schema_version'] = 'normalized_rrf_weight_ablation_v2_adaptive_followup'
summary_result['contract']['initial_predeclared_configurations'] = INITIAL_CONFIGURATIONS
summary_result['contract']['adaptive_exploratory_followup'] = {
    'trigger': 'Requested after observing the initial vector_0.4_bm25_0.6 development result.',
    'configurations': FOLLOWUP_CONFIGURATIONS,
    'same_retrieval_contract': True,
    'not_predeclared_with_initial_five': True,
    'cannot_retroactively_change_initial_selection': True,
}
summary_result['contract']['all_compared_configurations'] = CONFIGURATIONS
summary_result['results']['summaries'] = summary_rows
summary_result['results']['paired_evidence'] = paired_comparisons
summary_result['results']['candidate_recall_aggregate'] = candidate_recall_aggregate
summary_result['selection'] = INITIAL_SELECTION
summary_result['adaptive_exploratory_followup'] = {
    'provenance': 'Post-result adaptive exploratory test requested after the initial 0.4:0.6 result was known.',
    'new_configurations': list(FOLLOWUP_CONFIGURATIONS),
    'reference_baseline': BASELINE_CONFIGURATION,
    'reference_best_observed_from_initial_phase': 'vector_0.4_bm25_0.6',
    'extended_qualification': extended_qualification,
    'extended_best_observed_configuration': extended_best_observed,
    'extended_best_observed_gate_pass': extended_qualification[extended_best_observed]['qualified'],
    'followup_qualified_configurations': followup_qualified,
    'reference_decision': followup_decision,
    'interpretation': 'Passing would only nominate an exploratory candidate for a confirmatory retest; it cannot alter the initial automatic selection.',
}
summary_result['baseline_reproduction'].update({
    'initial_0.5_reference_rows_preserved_exact': True,
    'initial_0.4_reference_rows_preserved_exact': True,
    'initial_five_per_query_summary_candidate_rows_preserved_exact': True,
})
summary_result['integrity']['initial_phase_output_hashes_before_followup'] = INITIAL_OUTPUT_HASHES
summary_result['limitations'].append('The two BM25-heavy weights were chosen adaptively after inspecting the initial 0.4:0.6 result, so their estimates are exploratory and require a separately designed confirmatory retest.')
performance_result_hashes_after = {name: sha256_file(result_path) for name, result_path in PERFORMANCE_RESULT_FILES.items()}
performance_hash_exact_by_file = {
    name: performance_result_hashes_after[name] == frozen_hash
    for name, frozen_hash in FROZEN_SEVEN_WEIGHT_PERFORMANCE_HASHES.items()
}
assert performance_hash_exact_by_file['rrf_weight_ablation_per_query.csv']
assert performance_hash_exact_by_file['rrf_weight_ablation_summary.csv']
candidate_artifact_names = ('rrf_weight_candidate_recall.csv', 'rrf_weight_union_coverage.csv', 'rrf_weight_candidates.csv')
candidate_artifacts_exact = all(performance_hash_exact_by_file[name] for name in candidate_artifact_names)
summary_result['baseline_reproduction']['seven_weight_top5_metric_artifacts_exact_after_integrity_fix'] = True
summary_result['baseline_reproduction']['seven_weight_candidate_artifacts_exact_after_integrity_fix'] = candidate_artifacts_exact
summary_result['integrity']['seven_weight_performance_hashes_before_integrity_fix'] = FROZEN_SEVEN_WEIGHT_PERFORMANCE_HASHES
summary_result['integrity']['seven_weight_performance_hashes_after_integrity_fix'] = performance_result_hashes_after
summary_result['integrity']['seven_weight_performance_hash_exact_by_file'] = performance_hash_exact_by_file
summary_result['integrity']['candidate_tail_reproduction_comparison'] = {
    'all_candidate_artifacts_exact': candidate_artifacts_exact,
    'previous_union_mean_unique_count': 79.16666666666667,
    'current_union_mean_unique_count': summary_result['results']['union_coverage']['mean_unique_count'],
    'strict_hit_rate_unchanged': summary_result['results']['union_coverage']['strict_hit_rate'] == 1.0,
    'mean_strict_recall_unchanged': summary_result['results']['union_coverage']['mean_strict_recall'] == 0.9666666666666667,
    'diagnosis': (
        'All seven-weight performance and candidate artifacts stayed byte-exact after the integrity fix.'
        if candidate_artifacts_exact else
        'Top-5 metric artifacts stayed byte-exact while only depth-50-derived artifacts changed under a fresh temporary snapshot client. With identical source file hashes and squared-L2 distances, this is consistent with approximate HNSW tail-order variability across fresh processes; the current fresh-run candidate artifacts are recorded as the new result.'
    ),
}
summary_result['integrity']['output_hashes'] = {
    name: sha256_file(OUTPUT_ROOT / name) for name in (
        'rrf_weight_ablation_per_query.csv', 'rrf_weight_ablation_summary.csv', 'rrf_weight_candidate_recall.csv',
        'rrf_weight_union_coverage.csv', 'rrf_weight_candidates.csv', 'README.md',
    )
}

integrity_stage_checks.append(assert_current_chroma(chroma_file_hashes_before, FROZEN_EXPECTED_CHROMA_TREE_HASH, 'before_adaptive_final_artifact_save'))
input_hashes_followup_after = {name: sha256_file(path) for name, path in ALL_INPUT_FILES.items()}
chroma_file_hashes_followup_after = file_hash_map(CHROMA_ROOT)
chroma_tree_hash_followup_after = tree_hash_from_file_hash_map(chroma_file_hashes_followup_after)
assert chroma_file_hashes_followup_after == chroma_file_hashes_before
assert chroma_tree_hash_followup_after == tree_hash(CHROMA_ROOT) == FROZEN_EXPECTED_CHROMA_TREE_HASH
with sqlite3.connect(f'file:{database}?mode=ro', uri=True) as connection:
    chroma_count_followup_after = connection.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]
assert input_hashes_followup_after == input_hashes_before
assert chroma_tree_hash_followup_after == chroma_tree_hash_before
assert chroma_count_followup_after == chroma_count_before == 327
summary_result['integrity'].update({
    'input_hashes_after': input_hashes_followup_after,
    'chroma_file_hashes_before': chroma_file_hashes_before,
    'chroma_file_hashes_after': chroma_file_hashes_followup_after,
    'chroma_tree_hash_before': chroma_tree_hash_before,
    'chroma_tree_hash_after': chroma_tree_hash_followup_after,
    'chroma_tree_hash_algorithm': 'deterministic directory preorder (root files first, sorted names/subdirectories) + relative path bytes + raw SHA-256 digest bytes',
    'chroma_tree_hash_reconstructed_from_file_hash_map_exact': True,
    'frozen_expected_chroma_tree_hash_from_15': FROZEN_EXPECTED_CHROMA_TREE_HASH,
    'chroma_count_after': chroma_count_followup_after,
    'inputs_cache_chroma_unchanged': True,
    'integrity_check_stages': integrity_stage_checks,
})
atomic_json(OUTPUT_ROOT / 'rrf_weight_ablation_summary.json', summary_result)

final_manifest = json.loads((OUTPUT_ROOT / 'rrf_weight_run_manifest.json').read_text(encoding='utf-8'))
for name in ('rrf_weight_ablation_per_query.csv', 'rrf_weight_ablation_summary.csv', 'rrf_weight_ablation_summary.json', 'rrf_weight_candidate_recall.csv', 'rrf_weight_union_coverage.csv', 'rrf_weight_candidates.csv', 'README.md'):
    final_manifest['files'][f'output:{name}']['sha256'] = sha256_file(OUTPUT_ROOT / name)
final_manifest['files']['output:notebook']['sha256'] = 'pending_after_nbclient_serialization'
final_manifest['adaptive_exploratory_followup'] = {
    'new_configurations': list(FOLLOWUP_CONFIGURATIONS),
    'initial_automatic_decision_preserved': True,
}
atomic_json(OUTPUT_ROOT / 'rrf_weight_run_manifest.json', final_manifest)
integrity_stage_checks.append(assert_current_chroma(chroma_file_hashes_before, FROZEN_EXPECTED_CHROMA_TREE_HASH, 'after_last_artifact_save'))
print({
    'adaptive_followup': FOLLOWUP_CONFIGURATIONS,
    'followup_evidence_summaries': {configuration: summary_map[(configuration, 'evidence')] for configuration in FOLLOWUP_CONFIGURATIONS},
    'extended_best_observed': extended_best_observed,
    'extended_best_gate_pass': extended_qualification[extended_best_observed]['qualified'],
    'followup_qualified': followup_qualified,
    'followup_decision': followup_decision,
    'initial_decision_preserved': INITIAL_SELECTION['decision'],
    'rows': {'per_query': len(per_query_rows), 'summary': len(summary_rows), 'candidate_recall': len(candidate_recall_rows), 'candidates': len(candidate_rows)},
})

{'adaptive_followup': {'vector_0.3_bm25_0.7': (0.3, 0.7), 'vector_0.2_bm25_0.8': (0.2, 0.8)}, 'followup_evidence_summaries': {'vector_0.3_bm25_0.7': {'configuration': 'vector_0.3_bm25_0.7', 'vector_weight': 0.3, 'bm25_weight': 0.7, 'question_group': 'evidence', 'card_hit_at_3': 0.9, 'strict_evidence_hit_at_3': 0.7, 'recall_at_5': 0.7125, 'mrr_at_5': 0.6241666666666666, 'ndcg_at_5': 0.6080568615226605, 'denominator': 20, 'paired_mrr_wins': 3, 'paired_mrr_losses': 1, 'paired_mrr_ties': 16, 'paired_ndcg_wins': 3, 'paired_ndcg_losses': 3, 'paired_ndcg_ties': 14, 'guardrail_loss_query_count': 4}, 'vector_0.2_bm25_0.8': {'configuration': 'vector_0.2_bm25_0.8', 'vector_weight': 0.2, 'bm25_weight': 0.8, 'question_group': 'evidence', 'card_hit_at_3': 0.9, 'strict_evidence_hit_at_3': 0.7, 'recall_at_5': 0.7125, 'mrr_at_5': 0.5683333333333334, 'ndcg_at_5': 0.5764658569240395, 'denominator': 20, 'paired_mrr_wins': 4, 'paired_mrr_losses': 3, 'paired_mrr_ties': 13, 'paired_ndcg_wins': 4, 'paired_ndc

In [7]:
# Final validation after the adaptive follow-up rewrites the unified seven-weight artifacts.
stored_per_query = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_ablation_per_query.csv').open(encoding='utf-8')))
stored_summary = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_ablation_summary.csv').open(encoding='utf-8')))
stored_candidate_recall = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_candidate_recall.csv').open(encoding='utf-8')))
stored_union = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_union_coverage.csv').open(encoding='utf-8')))
stored_candidates = list(csv.DictReader((OUTPUT_ROOT / 'rrf_weight_candidates.csv').open(encoding='utf-8')))
stored_json = json.loads((OUTPUT_ROOT / 'rrf_weight_ablation_summary.json').read_text(encoding='utf-8'))
assert (len(stored_per_query), len(stored_summary), len(stored_candidate_recall), len(stored_union), len(stored_candidates)) == (210, 42, 210, 30, 10500)
assert set(row['configuration'] for row in stored_per_query) == set(CONFIGURATIONS)
assert all(sum(row['configuration'] == configuration for row in stored_per_query) == 30 for configuration in CONFIGURATIONS)
assert all(sum(row['configuration'] == configuration for row in stored_summary) == 6 for configuration in CONFIGURATIONS)
assert all(sum(row['configuration'] == configuration for row in stored_candidate_recall) == 30 for configuration in CONFIGURATIONS)
assert all(sum(row['configuration'] == configuration for row in stored_candidates) == 1500 for configuration in CONFIGURATIONS)
assert stored_json['selection']['decision'] == 'retain_baseline_0.5_0.5'
assert stored_json['selection']['selected_configuration'] == BASELINE_CONFIGURATION
assert stored_json['adaptive_exploratory_followup']['provenance'].startswith('Post-result adaptive exploratory')
assert stored_json['baseline_reproduction']['initial_0.5_reference_rows_preserved_exact']
assert stored_json['baseline_reproduction']['initial_0.4_reference_rows_preserved_exact']
assert stored_json['baseline_reproduction']['baseline_rrf_top5_per_query_summary_exact']
assert stored_json['vector_reproduction']['snapshot_published_top5_exact_queries'] == 30
assert stored_json['execution']['network_calls'] == stored_json['execution']['api_calls'] == stored_json['execution']['new_embeddings'] == 0
assert stored_json['integrity']['inputs_cache_chroma_unchanged']
assert stored_json['integrity']['chroma_tree_hash_before'] == FROZEN_EXPECTED_CHROMA_TREE_HASH
assert stored_json['integrity']['chroma_tree_hash_after'] == FROZEN_EXPECTED_CHROMA_TREE_HASH
assert stored_json['integrity']['frozen_expected_chroma_tree_hash_from_15'] == FROZEN_EXPECTED_CHROMA_TREE_HASH
assert stored_json['integrity']['chroma_tree_hash_reconstructed_from_file_hash_map_exact']
assert stored_json['integrity']['chroma_file_hashes_before'] == chroma_file_hashes_before
assert stored_json['integrity']['chroma_file_hashes_after'] == chroma_file_hashes_before
assert stored_json['baseline_reproduction']['seven_weight_top5_metric_artifacts_exact_after_integrity_fix']
assert stored_json['integrity']['seven_weight_performance_hash_exact_by_file']['rrf_weight_ablation_per_query.csv']
assert stored_json['integrity']['seven_weight_performance_hash_exact_by_file']['rrf_weight_ablation_summary.csv']
assert stored_json['integrity']['candidate_tail_reproduction_comparison']['current_union_mean_unique_count'] == stored_json['results']['union_coverage']['mean_unique_count']
stored_manifest = json.loads((OUTPUT_ROOT / 'rrf_weight_run_manifest.json').read_text(encoding='utf-8'))
manifest_chroma_hashes = {
    name.removeprefix('input:chroma:'): item['sha256']
    for name, item in stored_manifest['files'].items() if name.startswith('input:chroma:')
}
assert manifest_chroma_hashes == chroma_file_hashes_before
assert_current_chroma(chroma_file_hashes_before, FROZEN_EXPECTED_CHROMA_TREE_HASH, 'final_validation')
for row in stored_candidate_recall:
    assert int(row['candidate_count_at_10']) <= int(row['candidate_count_at_20']) <= int(row['candidate_count_at_50'])
    assert float(row['strict_recall_at_10']) <= float(row['strict_recall_at_20']) + TOLERANCE <= float(row['strict_recall_at_50']) + 2 * TOLERANCE
for row in stored_candidates:
    assert 1 <= int(row['fused_rank']) <= 50
    assert math.isclose(float(row['vector_score']) + float(row['bm25_score']), float(row['total_score']), rel_tol=0, abs_tol=1e-18)
assert json.loads((OUTPUT_ROOT / 'rrf_weight_run_manifest.json').read_text(encoding='utf-8'))['self_hash_excluded']
print({'seven_weight_validation': True, 'per_query_rows': len(stored_per_query), 'summary_rows': len(stored_summary), 'candidate_recall_rows': len(stored_candidate_recall), 'union_rows': len(stored_union), 'candidate_rows': len(stored_candidates), 'initial_selection_preserved': True, 'baseline_exact': True, 'top5_metric_results_exact_after_integrity_fix': True, 'candidate_artifacts_exact_after_integrity_fix': stored_json['baseline_reproduction']['seven_weight_candidate_artifacts_exact_after_integrity_fix'], 'frozen_chroma_tree_hash': FROZEN_EXPECTED_CHROMA_TREE_HASH, 'tree_hash_from_file_map_exact': True, 'manifest_individual_chroma_hashes_exact': True, 'source_unchanged': True, 'network_api_new_embeddings': 0})

{'seven_weight_validation': True, 'per_query_rows': 210, 'summary_rows': 42, 'candidate_recall_rows': 210, 'union_rows': 30, 'candidate_rows': 10500, 'initial_selection_preserved': True, 'baseline_exact': True, 'top5_metric_results_exact_after_integrity_fix': True, 'candidate_artifacts_exact_after_integrity_fix': False, 'frozen_chroma_tree_hash': '255dd5d9cdd84065a0d75a73b6b3b9bf09962d02797c4103352a909d4efc15c5', 'tree_hash_from_file_map_exact': True, 'manifest_individual_chroma_hashes_exact': True, 'source_unchanged': True, 'network_api_new_embeddings': 0}
